# Deep Learning 044 — Pooling

Pooling answers two problems at once. **Memory:** one convolutional layer on a
228×228 image with 100 filters produces feature maps costing **19.5 MiB per
example** and **0.61 GiB for a batch of 32** — 2×2 pooling cuts that **4×**.
**Translation:** convolution moves its answer when the input moves; pooling does
not.

| Part | What we check |
|---|---|
| A | the memory arithmetic, exactly |
| B | what 2×2 pooling saves |
| C | translation: conv output moves, pooled output does not |
| D | max against average pooling |

In [ ]:
import numpy as np
np.set_printoptions(linewidth=120, suppress=True)

def convolve(img, k, stride=1):
    kh, kw = k.shape
    oh = (img.shape[0] - kh) // stride + 1
    ow = (img.shape[1] - kw) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(img[i*stride:i*stride+kh, j*stride:j*stride+kw] * k)
    return out

def max_pool(x, k=2, stride=2):
    oh = (x.shape[0] - k) // stride + 1
    ow = (x.shape[1] - k) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = x[i*stride:i*stride+k, j*stride:j*stride+k].max()
    return out

## Part A — Why memory is the first problem

In [ ]:
h = w = 226          # 228x228 image, 3x3 filter, no padding
filters, batch, bytes_per = 100, 32, 4        # float32

per_example = h * w * filters * bytes_per
print(f"one example : {h}x{w}x{filters} x {bytes_per} bytes = {per_example:,} bytes"
      f" = {per_example/2**20:.1f} MiB")
print(f"batch of {batch} : {per_example*batch/2**30:.2f} GiB")
assert round(per_example / 2**20, 1) == 19.5
assert round(per_example * batch / 2**30, 2) == 0.61
print("\nAnd that is ONE layer's output, before any gradients are stored.")

## Part B — What pooling saves

In [ ]:
pooled = (h // 2) * (w // 2) * filters * bytes_per
print(f"after 2x2 pooling, stride 2: {pooled/2**20:.1f} MiB per example")
print(f"saving: {per_example/pooled:.0f}x")
assert round(per_example / pooled) == 4
print("\nIt keeps one value in four, so it divides the map by 4.")

## Part C — Translation

Shift the image by one pixel and compare what the convolution says with what the
pooled output says.

In [ ]:
img = np.zeros((8, 8)); img[2:5, 2:5] = 1      # a small bright square
shifted = np.zeros((8, 8)); shifted[3:6, 2:5] = 1   # the same square, one row down

k = np.ones((3, 3))
fa, fb = convolve(img, k), convolve(shifted, k)
pa, pb = max_pool(fa), max_pool(fb)

print("conv maps identical? ", np.array_equal(fa, fb))
print("pooled maps identical?", np.array_equal(pa, pb))
changed_conv = int((fa != fb).sum())
changed_pool = int((pa != pb).sum())
print(f"\npositions that changed: conv {changed_conv} of {fa.size}, "
      f"pooled {changed_pool} of {pa.size}")
assert changed_conv > changed_pool
print("\nPooling does not make the network blind to position - it widens the")
print("window in which 'the feature is here' counts as the same answer.")

## Part D — Max or average?

In [ ]:
patch = np.array([[1.0, 0.1], [0.2, 0.0]])
print("patch:\n", patch)
print("max    ->", patch.max(), " (did this feature appear anywhere?)")
print("average->", patch.mean(), " (how much of it was there on average?)")
print("\nA strong, local feature survives max pooling and is diluted by averaging:")
print(f"  max keeps {patch.max()/patch.max():.0%} of the peak, average keeps "
      f"{patch.mean()/patch.max():.0%}")

## What to take away

- One conv layer's output: **19.5 MiB per example**, **0.61 GiB for a batch of 32**.
- **2×2 pooling, stride 2, cuts that 4×** by keeping one value in four.
- Convolution is **translation-variant**; pooling gives tolerance to small shifts.
- **Max pooling asks "did this feature appear?"** — usually the right question.
- The conv layer is lesson 041's simple cell; the pooling layer is the complex cell.

## Exercises

1. Recompute Part A for a batch of 128. At what batch size does one layer pass 2 GiB?
2. Shift the square by two rows instead of one. Does 2×2 pooling still give the
   same answer? What about 4×4 pooling?
3. Replace max with average pooling in Part C. Which is more tolerant to the shift?